# Models, messages and streaming

## Goal

Create a model, inspect AIMessage, build an agent, pass explicit history and stream a reply.

In [ ]:
from pathlib import Path
import sys

# Find this course's root, whether Jupyter starts at the root or in notebooks/.
current_dir = Path.cwd().resolve()
course_root = next(
    (folder for folder in (current_dir, *current_dir.parents)
     if (folder / "course.json").is_file() and (folder / "lesson_support.py").is_file()),
    None,
)
if course_root is None:
    raise RuntimeError("Start Jupyter from the course root, as described in README.md.")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from lesson_support import make_model


## 1. Create a model and make a direct request

In [ ]:
# Creating the object does not make an API request.
model = make_model()
print("Model:", model.model_name)

In [ ]:
response = model.invoke("What's the capital of the Moon?")

response

In [ ]:
from pprint import pprint

print(type(response))
pprint([name for name in dir(response) if not name.startswith("_")])
print(type(response.response_metadata))
# For full documentation, run help(type(response)) in a separate cell.


In [ ]:
print(response.content)

In [ ]:
from pprint import pprint

pprint(response.response_metadata)
# Normalized usage works across supported integrations and API modes.
pprint(response.usage_metadata)

## 2. Configure the model

In [ ]:
# Create another model object with a shorter output limit.
short_model = make_model(max_tokens=256)
response = short_model.invoke("What's the capital of the Moon? Answer briefly.")
print(response.content)


## 3. Create an agent using the same model

In [ ]:
from langchain.agents import create_agent

agent = create_agent(model=model)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?")]}
)

In [ ]:
from pprint import pprint

pprint(response)

In [ ]:
print(response['messages'][-1].content)

## 4. Pass conversation history explicitly

We supply three messages ourselves. No checkpointer is connected here.
Luna City is a deliberately fictional example: we create the AIMessage ourselves.
`model.invoke` returns an AIMessage; `agent.invoke` returns a state dictionary with `messages`.
Use dot notation for message attributes and brackets for dictionary keys.


In [ ]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?"),
    AIMessage(content="The capital of the Moon is Luna City."),
    HumanMessage(content="Interesting, tell me more about Luna City")]}
)

pprint(response)

## 5. Stream a response

In [ ]:
# stream yields message chunks, not necessarily individual tokens or separate API calls.
for chunk, metadata in agent.stream(
    {"messages": [HumanMessage(content="Invent a fictional Moon capital called Luna City. Describe it briefly.")]},
    stream_mode="messages",
):
    if chunk.content:
        print(chunk.content, end="", flush=True)
print()
